# 01 — Market Data Providers

Entry point to the abstract data layer. Fetches daily crypto OHLCV from Yahoo via
`YahooFinanceProvider`, then reshapes the MultiIndex `(symbol, timestamp)` bar frame
into the two layouts the rest of the platform consumes.

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalTradePerformance       (iid bets)
      (bars)            (signal cols)   (bt_result)    └─  SignalAllocationPerformance (capital deployed)
        ▲
   this notebook
```

**What this notebook covers**

1. **Provider instantiation** — `YahooFinanceProvider` with optional `DataCache`
   (Parquet on disk) and `ProviderRegistry` for priority-based fallback.
2. **`get_bars()`** — daily OHLCV for a basket of crypto symbols, returned as a
   long-format `(symbol, timestamp)` MultiIndex DataFrame.
3. **Wide reshape** — `bars["close"].unstack("symbol")` to get the
   `(date × symbol)` close-price matrix used by the engine path.
4. **`get_returns()`** — convenience helper that returns a wide returns matrix
   directly, skipping the manual reshape.

In [1]:
from hailmary.data import DataCache, ProviderRegistry
from hailmary.data.providers import YahooFinanceProvider
from hailmary.data.base import Timeframe
import pandas as pd
pd.set_option('display.max_columns', None)


cache = DataCache('../../data/cache', ttl_hours=48)
# yahoo = YahooFinanceProvider(cache=cache)
yahoo = YahooFinanceProvider(cache=None)

# registry = ProviderRegistry()
# registry.register(yahoo, priority=10)
# print(registry)

In [2]:
# Fetch daily OHLCV for a basket of equities
symbols = [
    "BTC-USD",  # Bitcoin
    "ETH-USD",  # Ethereum
    "SOL-USD",  # Solana
    "BNB-USD",  # BNB
    # "XRP-USD",  # XRP
    # "ADA-USD",  # Cardano
    # "DOGE-USD", # Dogecoin
    # "AVAX-USD", # Avalanche
    # "LINK-USD", # Chainlink
    # "LTC-USD",  # Litecoin
]
start=pd.to_datetime('2022-01-01')
end=pd.to_datetime('2024-01-01')
bars = yahoo.get_bars(symbols, start=start, end=end)
assert len(bars.index.get_level_values('symbol').unique()) == len(symbols)
assert set(bars.index.get_level_values('symbol').unique()) == set(symbols)
assert bars.index.get_level_values("timestamp").min().strftime("%Y-%m-%d") == start.strftime("%Y-%m-%d")
assert bars.index.get_level_values("timestamp").max().strftime("%Y-%m-%d") == end.strftime("%Y-%m-%d")

print(bars.shape)
bars.head(4)

2026-04-26 22:36:57.338 | DEBUG    | hailmary.data.cache:get:38 - Cache hit key=dfade89a2257


(2924, 5)


open        high         low       close      volume
symbol  timestamp                                                             
BNB-USD 2022-01-01  511.910370  527.352722  511.903534  527.352722  1622547014
        2022-01-02  527.291626  533.371033  518.753540  531.396667  1462276185
        2022-01-03  531.388855  532.095581  510.652161  512.135986  1949153130
        2022-01-04  512.130310  519.701660  503.215363  507.506104  2200879165

In [3]:
# Wide close-price DataFrame
close = bars['close'].unstack(level=0)
close.tail()

symbol,BNB-USD,BTC-USD,ETH-USD,SOL-USD
timestamp,,,,
2023-12-28,323.598999,42627.855469,2347.566162,102.104568
2023-12-29,313.878754,42099.402344,2300.690674,106.311516
2023-12-30,317.166199,42156.902344,2292.065430,101.845085
2023-12-31,312.435699,42265.187500,2281.471191,101.505821
2024-01-01,314.408295,44167.332031,2352.327881,109.508682


In [4]:
# Convenience: get returns directly
returns = yahoo.get_returns(symbols,
                            start=pd.Timestamp("2022-01-01"),
                            end=pd.Timestamp("2024-01-01"))
returns.describe()

2026-04-26 22:37:20.377 | INFO     | hailmary.data.providers.yahoo:get_bars:64 - Fetching 4 symbols from Yahoo (2022-01-01 00:00:00 → 2024-01-01 00:00:00); inclusive
2026-04-26 22:37:21.160 | DEBUG    | hailmary.data.cache:set:45 - Cached 2924 rows key=f7e1ccca10b1


symbol,BNB-USD,BTC-USD,ETH-USD,SOL-USD
count,730.000000,730.000000,730.000000,730.000000
mean,-0.000205,0.000308,0.000020,0.000992
std,0.031495,0.028677,0.036390,0.057186
min,-0.185654,-0.159747,-0.174564,-0.422809
25%,-0.013478,-0.011264,-0.016147,-0.030118
50%,0.000462,-0.000429,-0.000503,-0.002011
75%,0.013107,0.013133,0.016321,0.029894
max,0.139503,0.145412,0.181149,0.325944
